In [ ]:
# =============================================================================
# Cross-level comparison: do individual-level metrics predict group-level significance?
# =============================================================================
# For each (Age × Chrom × Parcel), compute individual-level summaries from
# the per-infant TFCE results, then test whether these predict group-level
# significance from the cross-participant t-tests.
# =============================================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor

projectRoot = Path().resolve().parent

tablesDir = projectRoot / 'data'

# --- Load data ---
# Individual-level results (per infant × parcel × chrom × age)
indDf = pd.read_csv(tablesDir / "surf_parcel_TFCE_circshift_results_INDIV_spatiotemporal_FDR.csv")

# Group-level results (significant parcels only — need to reconstruct full set)
grpSigDf = pd.read_csv(tablesDir / "surf_parcel_t_test_results_corrected_SIGONLY.csv")

# --- Build individual-level summaries per (Age, Chrom, Parcel) ---
ind_summary = (
    indDf
    .groupby(['Age', 'Chrom', 'Parcel'])
    .agg(
        n_infants_tested=('Infant', 'nunique'),
        n_infants_sig=('Significant', 'sum'),
        median_ind_t=('t peak', 'median'),
        mean_ind_t=('t peak', 'mean'),
        median_ind_tfce=('TFCE peak', 'median'),
    )
    .reset_index()
)
ind_summary['pct_sig'] = 100.0 * ind_summary['n_infants_sig'] / ind_summary['n_infants_tested']

# --- Build group-level significance flag per (Age, Chrom, Parcel) ---
# grpSigDf only contains significant parcels; parcels absent from it are non-significant
grp_sig_set = set(
    zip(grpSigDf['Age'], grpSigDf['Chrom'], grpSigDf['Parcel'])
)
ind_summary['group_sig'] = ind_summary.apply(
    lambda r: (r['Age'], r['Chrom'], r['Parcel']) in grp_sig_set, axis=1
)

print(f"Total (Age, Chrom, Parcel) cells: {len(ind_summary)}")
print(f"  Group-significant: {ind_summary['group_sig'].sum()}")
print(f"  Group-non-significant: {(~ind_summary['group_sig']).sum()}")
print()

# --- Per age × chrom breakdown ---
for (age, chrom), sub in ind_summary.groupby(['Age', 'Chrom']):
    n_sig = sub['group_sig'].sum()
    n_total = len(sub)
    print(f"  Age {age}, Chrom {chrom}: {n_sig}/{n_total} parcels group-significant")

In [ ]:
# =============================================================================
# Analysis 1: Logistic regression
# Does individual-level signal predict group-level significance?
# Binary outcome: group_sig (True/False)
# Predictors: n_infants_sig, median_ind_t, pct_sig
# =============================================================================

print("=" * 100)
print("LOGISTIC REGRESSION: individual-level metrics predicting group-level significance")
print("=" * 100)

for (age, chrom), sub in ind_summary.groupby(['Age', 'Chrom']):
    n_sig = sub['group_sig'].sum()
    n_nonsig = (~sub['group_sig']).sum()

    # Need at least some group-sig parcels to fit logistic regression
    if n_sig < 3:
        print(f"\n  Age {age}, Chrom {chrom}: only {n_sig} group-sig parcels — "
              f"skipping logistic regression (insufficient positive cases)")
        continue

    print(f"\n{'=' * 80}")
    print(f"  Age {age}, Chrom {chrom} ({n_sig} group-sig, {n_nonsig} group-nonsig)")
    print(f"{'=' * 80}")

    y = sub['group_sig'].astype(int).values

    # Predictors: n_infants_sig, median_ind_t, pct_sig
    X = sub[['n_infants_sig', 'median_ind_t', 'pct_sig', 'median_ind_tfce']].copy()

    # Check for NaN/inf and drop
    valid_mask = X.notna().all(axis=1) & np.isfinite(X).all(axis=1)
    X = X[valid_mask]
    y = y[valid_mask.values]

    if len(y) < 10 or y.sum() < 3:
        print(f"  Insufficient data after filtering (n={len(y)}, n_sig={y.sum()}) — skipping")
        continue


    # Calculate VIF for each predictor (excluding the constant)
    X_no_const = sub[['n_infants_sig', 'median_ind_t', 'pct_sig', 'median_ind_tfce']].copy()
    X_no_const = X_no_const[X_no_const.notna().all(axis=1) & np.isfinite(X_no_const).all(axis=1)]

    print("  Variance Inflation Factors:")
    for i, col in enumerate(X_no_const.columns):
        vif = variance_inflation_factor(X_no_const.values, i)
        flag = ' *** HIGH' if vif > 5 else ''
        print(f"    {col}: VIF = {vif:.2f}{flag}")
    print()

    X = sm.add_constant(X)

    try:
        logit_model = sm.Logit(y, X).fit(disp=0)
        print(logit_model.summary())
        print()

        # Odds ratios
        print("  Odds ratios (exp(coef)):")
        for name, coef in zip(logit_model.params.index, logit_model.params):
            if name == 'const':
                continue
            print(f"    {name}: OR = {np.exp(coef):.4f} "
                  f"(95% CI [{np.exp(logit_model.conf_int().loc[name, 0]):.4f}, "
                  f"{np.exp(logit_model.conf_int().loc[name, 1]):.4f}])")
    except Exception as e:
        print(f"  Logistic regression failed: {e}")

In [ ]:
# =============================================================================
# Analysis 2: Mann-Whitney U comparison
# Compare individual-level metrics at group-sig vs group-nonsig parcels
# =============================================================================

print("=" * 100)
print("MANN-WHITNEY U: individual-level metrics at group-sig vs group-nonsig parcels")
print("=" * 100)

metrics_to_compare = [
    ('n_infants_tested', 'N infants tested'),
    ('n_infants_sig', 'N infants individually significant'),
    ('pct_sig', '% infants individually significant'),
    ('median_ind_t', 'Median individual t-statistic'),
    ('median_ind_tfce', 'Median individual TFCE peak'),
]

for (age, chrom), sub in ind_summary.groupby(['Age', 'Chrom']):
    sig = sub[sub['group_sig'] == True]
    nonsig = sub[sub['group_sig'] == False]

    if len(sig) < 3:
        print(f"\n  Age {age}, Chrom {chrom}: only {len(sig)} group-sig parcels — skipping")
        continue

    print(f"\n{'=' * 80}")
    print(f"  Age {age}, Chrom {chrom} (n_group_sig = {len(sig)}, n_group_nonsig = {len(nonsig)})")
    print(f"{'=' * 80}")

    for col, label in metrics_to_compare:
        v_sig = sig[col].dropna()
        v_nonsig = nonsig[col].dropna()

        if len(v_sig) < 3 or len(v_nonsig) < 3:
            print(f"  {label}: insufficient data (sig n={len(v_sig)}, nonsig n={len(v_nonsig)})")
            continue

        # One-sided: H1 = group-sig parcels have HIGHER individual-level metrics
        u, p_val = mannwhitneyu(v_sig, v_nonsig, alternative='greater')

        print(f"  {label}:")
        print(f"    Group-sig parcels:     median = {v_sig.median():.4f} (n = {len(v_sig)})")
        print(f"    Group-nonsig parcels:  median = {v_nonsig.median():.4f} (n = {len(v_nonsig)})")
        print(f"    Mann-Whitney U = {u:.1f}, p = {p_val:.4g} (H1: group-sig > group-nonsig)")
        print()

In [ ]:
# =============================================================================
# Multiple comparison correction for logistic regression and Mann-Whitney U
# Bonferroni and Holm-Bonferroni, corrected within each age × chrom
# =============================================================================

# --- Logistic regression: correct across predictor coefficients within each model ---

print("=" * 100)
print("LOGISTIC REGRESSION — corrected across predictor coefficients within each age × chrom")
print("=" * 100)

predictor_cols = ['n_infants_sig', 'median_ind_t', 'pct_sig', 'median_ind_tfce']

for (age, chrom), sub in ind_summary.groupby(['Age', 'Chrom']):
    n_sig = sub['group_sig'].sum()
    if n_sig < 3:
        print(f"\n  Age {age}, Chrom {chrom}: only {n_sig} group-sig parcels — skipping")
        continue

    y = sub['group_sig'].astype(int).values
    X = sub[predictor_cols].copy()
    valid_mask = X.notna().all(axis=1) & np.isfinite(X).all(axis=1)
    X = X[valid_mask]
    y = y[valid_mask.values]

    if len(y) < 10 or y.sum() < 3:
        print(f"\n  Age {age}, Chrom {chrom}: insufficient data — skipping")
        continue

    X_with_const = sm.add_constant(X)

    try:
        logit_model = sm.Logit(y, X_with_const).fit(disp=0)
    except Exception as e:
        print(f"\n  Age {age}, Chrom {chrom}: logistic regression failed — {e}")
        continue

    # Extract predictor p-values (exclude constant)
    coef_names = [n for n in logit_model.pvalues.index if n != 'const']
    raw_pvals = [logit_model.pvalues[n] for n in coef_names]
    coefs = [logit_model.params[n] for n in coef_names]

    n_tests = len(raw_pvals)
    _, bonf_p, _, _ = multipletests(raw_pvals, alpha=0.05, method='bonferroni')
    _, holm_p, _, _ = multipletests(raw_pvals, alpha=0.05, method='holm')

    print(f"\n{'=' * 80}")
    print(f"  Age {age}, Chrom {chrom} (n_tests = {n_tests} predictors)")
    print(f"{'=' * 80}")
    print(f"  {'Predictor':<25s} {'coef':>8s} {'raw p':>10s} {'Bonf p':>10s} {'Holm p':>10s}  Bonf sig  Holm sig")
    for name, coef, raw, bp, hp in zip(coef_names, coefs, raw_pvals, bonf_p, holm_p):
        b_flag = '✓' if bp < 0.05 else '—'
        h_flag = '✓' if hp < 0.05 else '—'
        print(f"  {name:<25s} {coef:>+8.4f} {raw:>10.4g} {bp:>10.4g} {hp:>10.4g}  {b_flag:>8s}  {h_flag:>8s}")

# --- Mann-Whitney U: correct across 5 metrics within each age × chrom ---

print("\n\n" + "=" * 100)
print("MANN-WHITNEY U — corrected across the 4 core metrics within each age × chrom")
print("=" * 100)

# NOTE: n_infants_tested is deliberately NOT included in the corrected battery
# below - the paper treats it as a separate control check (confirming sample
# size doesn't differ between group-significant and group-non-significant
# parcels), reported uncorrected, not as one of the four metrics whose
# p-values get Holm-Bonferroni-corrected together. See the standalone check
# further down.

metrics_to_compare = [
    ('n_infants_sig', 'N infants individually significant'),
    ('pct_sig', '% infants individually significant'),
    ('median_ind_t', 'Median individual t-statistic'),
    ('median_ind_tfce', 'Median individual TFCE peak'),
]

for (age, chrom), sub in ind_summary.groupby(['Age', 'Chrom']):
    sig = sub[sub['group_sig'] == True]
    nonsig = sub[sub['group_sig'] == False]

    if len(sig) < 3:
        print(f"\n  Age {age}, Chrom {chrom}: only {len(sig)} group-sig parcels — skipping")
        continue

    raw_pvals = []
    metric_names = []
    medians_sig = []
    medians_nonsig = []
    u_stats = []

    for col, label in metrics_to_compare:
        v_sig = sig[col].dropna()
        v_nonsig = nonsig[col].dropna()
        if len(v_sig) < 3 or len(v_nonsig) < 3:
            raw_pvals.append(np.nan)
            metric_names.append(label)
            medians_sig.append(np.nan)
            medians_nonsig.append(np.nan)
            u_stats.append(np.nan)
            continue
        u, p_val = mannwhitneyu(v_sig, v_nonsig, alternative='greater')
        raw_pvals.append(p_val)
        metric_names.append(label)
        medians_sig.append(v_sig.median())
        medians_nonsig.append(v_nonsig.median())
        u_stats.append(u)

    # Only correct across non-NaN p-values
    valid_idx = [i for i, p in enumerate(raw_pvals) if not np.isnan(p)]
    valid_pvals = [raw_pvals[i] for i in valid_idx]

    if not valid_pvals:
        print(f"\n  Age {age}, Chrom {chrom}: no valid tests — skipping")
        continue

    n_tests = len(valid_pvals)
    _, bonf_valid, _, _ = multipletests(valid_pvals, alpha=0.05, method='bonferroni')
    _, holm_valid, _, _ = multipletests(valid_pvals, alpha=0.05, method='holm')

    # Map corrected p-values back
    bonf_p = [np.nan] * len(raw_pvals)
    holm_p = [np.nan] * len(raw_pvals)
    for j, i in enumerate(valid_idx):
        bonf_p[i] = bonf_valid[j]
        holm_p[i] = holm_valid[j]

    print(f"\n{'=' * 80}")
    print(f"  Age {age}, Chrom {chrom} (n_group_sig = {len(sig)}, "
          f"n_group_nonsig = {len(nonsig)}, n_tests = {n_tests})")
    print(f"{'=' * 80}")
    print(f"  {'Metric':<40s} {'med(sig)':>9s} {'med(nonsig)':>12s} {'U':>8s} "
          f"{'raw p':>10s} {'Bonf p':>10s} {'Holm p':>10s}  Bonf  Holm")
    for i in range(len(metric_names)):
        if np.isnan(raw_pvals[i]):
            print(f"  {metric_names[i]:<40s} {'—':>9s} {'—':>12s} {'—':>8s} "
                  f"{'—':>10s} {'—':>10s} {'—':>10s}  {'—':>4s}  {'—':>4s}")
            continue
        b_flag = '✓' if bonf_p[i] < 0.05 else '—'
        h_flag = '✓' if holm_p[i] < 0.05 else '—'
        print(f"  {metric_names[i]:<40s} {medians_sig[i]:>9.4f} {medians_nonsig[i]:>12.4f} "
              f"{u_stats[i]:>8.1f} {raw_pvals[i]:>10.4g} {bonf_p[i]:>10.4g} "
              f"{holm_p[i]:>10.4g}  {b_flag:>4s}  {h_flag:>4s}")

# --- Separate, UNCORRECTED control check: does n_infants_tested differ? ---
# Matches the paper's description: reported individually as a check that
# group-significant and group-non-significant parcels don't simply differ in
# how many infants were tested (i.e. ruling out a sample-size confound) -
# not part of the corrected battery of four metrics above.

print("\n\n" + "=" * 100)
print("MANN-WHITNEY U — N INFANTS TESTED (separate control check, uncorrected)")
print("=" * 100)

for (age, chrom), sub in ind_summary.groupby(['Age', 'Chrom']):
    sig = sub[sub['group_sig'] == True]
    nonsig = sub[sub['group_sig'] == False]

    if len(sig) < 3:
        print(f"\n  Age {age}, Chrom {chrom}: only {len(sig)} group-sig parcels — skipping")
        continue

    v_sig = sig['n_infants_tested'].dropna()
    v_nonsig = nonsig['n_infants_tested'].dropna()

    if len(v_sig) < 3 or len(v_nonsig) < 3:
        print(f"\n  Age {age}, Chrom {chrom}: insufficient data — skipping")
        continue

    u, p_val = mannwhitneyu(v_sig, v_nonsig, alternative='greater')

    print(f"\n  Age {age}, Chrom {chrom} (n_group_sig = {len(sig)}, n_group_nonsig = {len(nonsig)})")
    print(f"    N infants tested: median(sig) = {v_sig.median():.1f}, "
          f"median(nonsig) = {v_nonsig.median():.1f}")
    print(f"    Mann-Whitney U = {u:.1f}, p = {p_val:.4g} (uncorrected)")